<a href="https://colab.research.google.com/github/araoclaudio2-create/Assignment-AI-2/blob/main/Assignment_11_into_to_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

data_dir = "/content/images"  # update if needed

categories = os.listdir(data_dir)

img_size = 64

X = []
y = []

for category in categories:
    path = os.path.join(data_dir, category)
    label = categories.index(category)

    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)

        try:
            img = cv2.imread(img_path)
            img = cv2.resize(img, (img_size, img_size))
            img = img / 255.0
            X.append(img.flatten())
            y.append(label)
        except:
            pass

X = np.array(X)
y = np.array(y)

print("Total images:", len(X))
print("Categories:", categories)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [10, None]
}

rf = RandomForestClassifier()

grid = GridSearchCV(rf, param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

importances = best_model.feature_importances_

top = np.argsort(importances)[-10:]

plt.barh(range(len(top)), importances[top])
plt.title("Top 10 Important Features")
plt.show()

def predict_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (img_size, img_size))
    img = img / 255.0
    img = img.flatten().reshape(1, -1)

    pred = best_model.predict(img)
    return categories[pred[0]]

# Example
print(predict_image("/content/test.jpg"))